# Параметры моделей

Подразделяются на две категории:

1. **Внутренние (параметры модели)**

Подбираются во время обучения и определяют, как использовать входные данные для получения необходимого результата.

Например, это веса (коэффициенты уравнения) в линейной/логистической регрессии.

2. **Внешние (параметры алгоритма)**

Их принято называть гиперпараметрами. Внешние параметры могут быть произвольно установлены перед началом обучения и контролируют внутреннюю работу обучающего алгоритма.

Например, это параметр регуляризации в линейной/логистической регрессии.

**Гиперпараметры** отвечают за сложность взаимосвязи между входными признаками и целевой переменной, поэтому сильно влияют на модель и качество прогнозирования.

# Базовая оптимизация

Наиболее часто используемый метод — это поиск по сетке (grid search) в sklearn, который по сути является попыткой перебрать все возможные комбинации заданных гиперпараметров. Мы указываем список значений для различных гиперпараметров, и, ориентируясь на нашу метрику, оцениваем эффективность модели для каждого их сочетания, чтобы получить оптимальную комбинацию значений.

Допустим, мы хотим подобрать гиперпараметры min_samples_leaf и max_depth для алгоритма DecisionTreeClassifier. Зададим списки их значений:

```python
    min_samples_leaf = [3, 5, 8, 9]
    max_depth = [4, 5, 6, 7, 8]
```

Поскольку нам нужно перебрать четыре различных значения для min_samples_leaf и пять — для max_depth, то получается всего 4*5=20 комбинаций. Модель будет обучена 20 раз; столько же раз будет рассчитана метрика.

# Опасность переобучения и утечки данных

<img src='Images/ml_01.png'>

Рассмотренный метод разбиения данных на обучающий, проверочный и тестовый наборы является вполне рабочим и относительно широко используемым, но весьма чувствителен к равномерности разбиения данных.

Для лучшей оценки обобщающей способности вместо одного разбиения данных на обучающий и проверочный наборы мы можем воспользоваться перекрёстной проверкой, то есть кросс-валидацией (cross validation). В таком случае качество модели оценивается для каждой комбинации гиперпараметров по всем разбиениям кросс-валидации. 

<img src='Images/ml_02.png'>

**Пояснение к рисунку**. 

Предположим, что у нас есть n комбинаций гиперпараметров. Берём первую комбинацию и обучаем на них первую модель с помощью кросс-валидации с 10 фолдами (cv=10), затем рассчитываем метрику как среднее по всем разбиениям. Так проделываем для каждой комбинации и выбираем ту, при которой наша метрика наилучшая. В итоге мы обучим n*cv моделей, но выберем один набор гиперпараметров, который и будет использоваться для обучения итоговой модели на всей обучающей выборке.

# GridSearchCV

Поскольку поиск по сетке с кросс-валидацией является весьма распространённым методом настройки гиперпараметров, библиотека scikit-learn предлагает класс GridSearchCV, в котором осуществляется именно такой вариант.

# Пример: подбор параметров для RandomForestClassifier

```python
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.model_selection import GridSearchCV

    # 1. Модель
    model = RandomForestClassifier(random_state=42)

    # 2. Словарь параметров для перебора
    param_grid = {
        'n_estimators': [100, 200],
        'max_depth': [None, 10, 20],
        'min_samples_split': [2, 5]
    }

    # 3. Настройка GridSearch
    grid_search = GridSearchCV(
        estimator=model,
        param_grid=param_grid,
        scoring='accuracy',       # метрика
        cv=5,                     # кросс-валидация (5 фолдов)
        n_jobs=-1,                # параллельно
        verbose=2                 # уровень логов
    )

    # 4. Обучение
    grid_search.fit(X_train, y_train)

    # 5. Лучшие параметры и метрика
    print("Лучшие параметры:", grid_search.best_params_)
    print("Лучшая точность:", grid_search.best_score_)
```

# RandomizedSearchCV

Альтернативным подходом подбора различных комбинаций гиперпараметров в библиотеке scikit-learn является RandomizedSearchCV.

<img src='Images/ml_03.png'>

На этой картинке изображено принципиальное различие двух методов: 

* **В GridSearchCV** сетка задаётся вручную, перебираются различные значения гиперпараметров с каким-то шагом, в итоге получается что-то похожее на «красивую» сетку слева на картинке. Однако минимум функции (белое пятно) мы так и не обнаруживаем — а ведь он где-то рядом, возможно, просто между подобранными нами комбинациями.

* **RandomizedSearchCV** выбирает n (количество задаём сами) случайных точек/комбинаций из заданных нами последовательностей. Как следствие, мы можем перебирать не все возможные точки, а только часть из них, тем самым управляя скоростью работы перебора.

**Основные параметры RandomizedSearchCV** аналогичны GridSearchCV, за исключением наименований некоторых параметров и наличия параметра n_iter:

* estimator — алгоритм, который будем оптимизировать;
* param_distributions — cловарь с именами параметров (str) в качестве ключей и списками параметров в качестве значений, которые нужно попробовать.
* scoring — по умолчанию используется score-функция заданного алгоритма:
* cv — количество фолдов в кросс-валидации, по умолчанию используется 5.
* n_jobs — количество ядер для распараллеливания расчёта. -1 использует все существующие ядра.
* n_iter — количество комбинаций на расчёт. От этого параметра напрямую зависит время оптимизации и качество модели.

Параметры по-умолчанию для обоих классов:

* для классификации — sklearn.metrics.accuracy_score;
* для регрессии — sklearn.metrics.r2_score.

# Рекомендации по настройке гиперпараметров ансамблей над решающими деревьями

### Алгоритм случайного леса (RandomForest)

* n_estimators — число итераций (количество деревьев). Частично работает правило «чем больше, тем лучше», но иногда это не имеет особого смысла и сильно увеличивает затраты, поэтому стоит пробовать обучать сотни деревьев [100,200, 300, 400]. Если нет изменений, то оставить минимальное — 100.
* max_depth — максимальная глубина дерева. В случайном лесе строятся «сильные» деревья, каждое из которых даёт полноценный прогноз, поэтому глубина деревьем может быть достаточно большой. Стоит следить за переобучением.
* max_features — максимальное количество признаков, учитываемых алгоритмом при поиске лучшего разделения;
* max_samples — доля выборки, которая будет использоваться для обучения каждого алгоритма — дерева.

# Алгоритм градиентного бустинга (GradientBoosting)

* n_estimators — число итераций (количество деревьев) : хотя ошибка на обучении монотонно стремится к нулю, ошибка на контроле, как правило, начинает увеличиваться после определенной итерации. Оптимальное число итераций можно выбирать, например, по отложенной выборке или с помощью кросс-валидации.

* learning_rate — темп обучения (0;1]:

На практике оказывается, что градиентный бустинг очень быстро строит композицию, ошибка которой на обучении выходит на асимптоту (достигает предела), после чего начинает настраиваться на шум и переобучаться. Параметр learning_rate контролирует, насколько сильно каждое дерево будет пытаться исправить ошибки предыдущих деревьев. Более высокая скорость обучения означает, что каждое дерево может внести более сильные корректировки. Как правило, чем меньше темп обучения, тем лучше качество итоговой композиции.

* max_depth — максимальная глубина дерева. Используется для борьбы с переобучением. Рекомендуется устанавливать не более 5.
* max_features — максимальное количество признаков, учитываемых алгоритмом при поиске лучшего разделения.
* subsample — доля выборки, которая будет использоваться для обучения каждого алгоритма. Это ещё один способ улучшения качества градиентного бустинга. Таким образом вносится рандомизация в процесс обучения базовых алгоритмов, что снижает уровень шума в обучении, а также повышает эффективность вычислений. 

Рекомендация. Берите подвыборки, размер которых вдвое меньше исходной выборки.

Важно: Основные параметры градиентного бустинга деревьев — это количество деревьев (n_estimators) и скорость обучения (learning_rate), контролирующие степень вклада каждого дерева в устранение ошибок предыдущих деревьев. Эти два параметра тесно взаимосвязаны, поскольку более низкое значение learning_rate означает, что для построения модели аналогичной сложности необходимо большее количество деревьев.

**Общепринятая практика для бустинга** — подгонять n_estimators в зависимости от бюджета времени и памяти, а затем подбирать различные значения learning_rate.

# Продвинутая оптимизация

В реальных задачах оптимизация может занимать часы и даже сутки, поэтому в идеале необходимо оптимизировать гиперпараметры самым эффективным образом, чтобы снизить затраты и время на вычисления.

Один из способов  — это **байесовская оптимизация**. Она отличается от случайного поиска или поиска по сетке тем, что учитывает предыдущие результаты, а не выбирает комбинации из вариантов, не имеющих информации о прошлых оценках. Во многих случаях это позволяет найти лучшие значения гиперпараметров модели за меньшее количество времени. Таким образом, мы получаем и более быструю оптимизацию, и более качественный результат.

Существует несколько разных алгоритмов для этого типа оптимизации, но особенно используемым является **Tree-Structured Parzen Estimators (TPE)**.

# Tree-Structured Parzen Estimators (TPE)

**TPE** — это байесовский метод оптимизации гиперпараметров, основанный на построении вероятностных моделей, который оценивает и предсказывает, какие параметры с наибольшей вероятностью дадут лучший результат.

Он реализован, например, в библиотеке Optuna, Hyperopt и других.

### Как работает TPE?

1. Разделяет прошлые пробы на 2 группы:

* «Хорошие» (с низкой ошибкой / высокой метрикой)
* «Плохие» (все остальные)

2. Строит две модели распределения:

* l(x) — плотность вероятности параметров в «хорошей» зоне
* g(x) — в «плохой» зоне

Выбирает следующий набор гиперпараметров x так, чтобы максимизировать отношение: $argmax_x$ $l(x) / g(x)$

<img src='Images/ml_04.png'>

Таким образом ищет параметры, которые вероятнее всего дадут лучший результат, с учётом уже пройденных точек.

### Плюсы TPE:

* Эффективнее перебора (GridSearch) и случайного поиска (RandomSearch)

* Учитывает историю предыдущих экспериментов

* Хорошо работает с непрерывными, категориальными и вложенными параметрами

* Легко интегрируется с Optuna, Hyperopt

# Hyperopt

**Hyperopt** — это библиотека Python с открытым исходным кодом на основе байесовской оптимизации, в которой реализован алгоритм Tree-Structured Parzen Estimators (TPE).

### Три шага для использования Hyperopt:

1. Задание пространства поиска гиперпараметров. 

Объявляем список гиперпараметров, тип распределения и его границы.

### Основные (наиболее часто используемые) типы:

* hp.choice(label, options) — равновероятный выбор из массива. Массив (список/кортеж) вы задаёте сами, в списке могут быть как числа, так и строки (категории), но, как правило, данный метод используется для оптимизации категориального гиперпараметра (например, тип регуляризации в линейной регрессии или критерий информативности в деревьях);

* hp.randint(label, upper) — возвращает случайное целое число из диапазона [0, upper];

* hp.uniform(label, low, high) — создаёт равномерное непрерывное распределение и возвращает случайное число (не обязательно целое) из диапазона [low, high];

* hp.quniform(label, low, high, q) - возвращает дискретные целочисленные значения для параметра label в диапазоне между low и high с шагом q.

* hp.normal(label, mu, sigma) — создаёт нормальное непрерывное распределение с параметрами mu и sigma и возвращает случайное число из этого распределения;

* hp.lognormal(label, mu, sigma) — создаёт логнормальное непрерывное распределение с параметрами mu и sigma и возвращает случайное число из этого распределения.

2. Задание целевой функции. 

Создаём модель МО, передаём ей данные и оцениваем её на основе выбранной метрики. Можем минимизировать/максимизировать значение метрики.

3. Задание алгоритма поиска:

* Random Search.
* Tree of Parzen Estimators (TPE).

# Optuna

**Optuna** — это достаточно новый фреймворк/библиотека, разработанный специально для оптимизации гиперпараметров. Помимо байесовских алгоритмов, есть возможность удаления плохих комбинаций из рассмотрения. По умолчанию алгоритм удаляет комбинации, в которых модель даёт качество ниже медианы из уже рассмотренных. Optuna помогает  быстрее находить лучшие гиперпараметры и работает с большинством современных известных библиотек ML, таких как scikit-learn, xgboost, PyTorch, TensorFlow, skorch, lightgbm, Keras, fast-ai и другими.

### Три шага для использования Optuna: 

1. Задание пространства поиска гиперпараметров.

### Основные функции:

* suggest_categorical(name, choices) — для категориальных гиперпараметров;
* suggest_int(name, low, high, step=1, log=False) — для целочисленных гиперпараметров;
* suggest_float(name, low, high, step=None, log=False) — для непрерывных гиперпараметров;
* suggest_uniform(name, low, high) — для целочисленных и непрерывных гиперпараметров.

С помощью необязательных аргументов step и log можно дискретизировать или взять логарифм целочисленных и непрерывных параметров.

2. Задание целевой функции. 

Создаём модель МО, передаём ей данные и оцениваем её на основе выбранной метрики, можем минимизировать/максимизировать значение метрики. На данном этапе будет обучена модель только на одной комбинации гиперпараметров.

3. Создание объекта исследования **create study**. 

По умолчанию используется алгоритм поиска TPE (есть и другие варианты) и вызов метода optimize(), в который передаётся целевая функция, созданная на первом шаге. Выполняется заданное n_trials раз, подставляются различные комбинации гиперпараметров.

<img src='Images/ml_05.png'>